# Snowflake Cortex Workshop (Shortened)
## Free Dataset → Semantic View → Cortex Agent → Snowflake CoWork

We're going to build an AI agent that answers economic questions about the financial well being of Americans, then chat with it in Snowflake CoWork.

The key idea: **define your business logic once, in the data layer, not in prompts.** Every consumer gets the same correct answer from the same governed data.

**What you will build**
- A table of US economic indicators built from the free *Snowflake Public Data* Marketplace listing
- A **Semantic View** defining dimensions, facts, and metrics
- A **Cortex Agent** that turns natural-language questions into governed SQL
- A **Snowflake CoWork** chat experience for a business user who can only consume, not build

**Prerequisites**
- A Snowflake account with `ACCOUNTADMIN` access

**Components**
- **Snowflake Marketplace**: instant access to curated, live datasets. No ETL.
- **Semantic View**: business logic defined once in the data layer; grounds the AI so it doesn't guess.
- **Cortex Agent**: natural-language interface that turns questions into governed SQL and returns grounded answers.
- **Snowflake CoWork**: chat interface inside Snowflake for business users who don't write SQL.

## Step 1: Role-Based Access Control

Throughout this lab we use two roles:
- `hol_role`: this is us, the developer. It runs the lab and owns everything we create.
- `end_user_role`: simulates a business user who can only ask questions through the agent but can't build or modify anything.

Run the cell below to create the warehouse, both roles, and the needed privileges.

In [ ]:
USE ROLE ACCOUNTADMIN;

-- Create a warehouse for this lab (guaranteed to exist)
CREATE WAREHOUSE IF NOT EXISTS hol_wh
  WAREHOUSE_SIZE = 'XSMALL' AUTO_SUSPEND = 60 INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE hol_wh;

-- Builder role: owns all workshop objects
CREATE ROLE IF NOT EXISTS hol_role;
-- Consumer role: can only use the agent (CoWork)
CREATE ROLE IF NOT EXISTS end_user_role;

-- Attach both roles to the role hierarchy.
-- hol_role OWNS everything we build, so without this an admin has no path to
-- those objects once the workshop user goes away -- and cleanup becomes impossible.
-- ACCOUNTADMIN inherits SYSADMIN, so it can always do whatever hol_role can do.
GRANT ROLE hol_role TO ROLE SYSADMIN;
GRANT ROLE end_user_role TO ROLE SYSADMIN;

-- Grant both roles to whoever is running this lab
BEGIN
  LET usr := CURRENT_USER();
  EXECUTE IMMEDIATE 'GRANT ROLE hol_role TO USER ' || :usr;
  EXECUTE IMMEDIATE 'GRANT ROLE end_user_role TO USER ' || :usr;
  EXECUTE IMMEDIATE 'ALTER USER ' || :usr || ' SET DEFAULT_WAREHOUSE = ''hol_wh''';
END;

-- Builder privileges
GRANT CREATE DATABASE  ON ACCOUNT TO ROLE hol_role;
GRANT CREATE WAREHOUSE ON ACCOUNT TO ROLE hol_role;
GRANT OWNERSHIP ON WAREHOUSE hol_wh TO ROLE hol_role COPY CURRENT GRANTS;

-- Cortex access for both roles
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE hol_role;
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE end_user_role;

-- Switch to hol_role to build
USE ROLE hol_role;
USE WAREHOUSE hol_wh;

CREATE DATABASE IF NOT EXISTS hol_db;
USE SCHEMA hol_db.public;

-- Schema-level privileges (must come after database/schema exist)
USE ROLE ACCOUNTADMIN;
GRANT CREATE SEMANTIC VIEW ON SCHEMA hol_db.public TO ROLE hol_role;
USE ROLE hol_role;
USE WAREHOUSE hol_wh;
USE SCHEMA hol_db.public;

-- Grant consumer role usage on warehouse and database
GRANT USAGE ON WAREHOUSE hol_wh TO ROLE end_user_role;
GRANT USAGE ON DATABASE hol_db TO ROLE end_user_role;
GRANT USAGE ON SCHEMA hol_db.public TO ROLE end_user_role;

-- Verify context
SELECT CURRENT_ROLE() AS role, CURRENT_WAREHOUSE() AS wh,
       CURRENT_DATABASE() AS db, CURRENT_SCHEMA() AS schema;

## Step 2: Get the Free Dataset from Snowflake Marketplace

We want an economic dataset that tracks the financial wellbeing of Americans at the state level: income, inflation, mortgage rates, and unemployment, all on a monthly basis. That means four source tables from the Bureau of Labor Statistics and related public data.

Snowflake Marketplace lets you access curated, live datasets instantly. You click **Get** and the data appears in your account. No ETL pipelines, no data copying.

**UI-Snowsight:** Marketplace → Snowflake Marketplace → Data products → search **"Snowflake Public Data"** → click **Snowflake Public Data (Free)** → **Get**.
- Database name: `SNOWFLAKE_PUBLIC_DATA_FREE` (accept the default options and don't change them).
- Direct link: https://app.snowflake.com/marketplace/listing/GZTSZ290BV255/snowflake-public-data-products-snowflake-public-data-free

This dataset is already available on many Snowflake accounts. The query below verifies access to all four source tables we need.

In [ ]:
-- Verify marketplace data access
SELECT 'BLS_PRICE' AS source, COUNT(*) AS row_count
  FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.BUREAU_OF_LABOR_STATISTICS_PRICE_TIMESERIES
UNION ALL
SELECT 'BLS_EMPLOYMENT', COUNT(*)
  FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.BUREAU_OF_LABOR_STATISTICS_EMPLOYMENT_TIMESERIES
UNION ALL
SELECT 'FREDDIE_MAC', COUNT(*)
  FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.FREDDIE_MAC_HOUSING_TIMESERIES
UNION ALL
SELECT 'IRS_INCOME', COUNT(*)
  FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.IRS_INDIVIDUAL_INCOME_TIMESERIES;

## Step 3: Build the Economic Indicators Table

Now we join the four marketplace sources into one wide-format table in `hol_db.public`. It holds national rows (CPI, inflation, mortgage rates, unemployment, income index) and state rows (unemployment and income index).

In [ ]:
-- Join four marketplace sources into one wide-format table
CREATE OR REPLACE TABLE hol_db.public.economic_indicators AS
WITH cpi AS (
  SELECT DATE_TRUNC('month', date) AS month, AVG(value) AS cpi_index
  FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.BUREAU_OF_LABOR_STATISTICS_PRICE_TIMESERIES
  WHERE variable_name = 'CPI: All items, Not seasonally adjusted, Monthly'
    AND geo_id = 'country/USA'
  GROUP BY 1
),
mortgage_30yr AS (
  SELECT DATE_TRUNC('month', date) AS month, ROUND(AVG(value) * 100, 2) AS mortgage_rate_30yr_pct
  FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.FREDDIE_MAC_HOUSING_TIMESERIES
  WHERE variable_name = '30-Year Fixed Rate Mortgage Rate, National Average'
    AND geo_id = 'country/USA'
  GROUP BY 1
),
mortgage_15yr AS (
  SELECT DATE_TRUNC('month', date) AS month, ROUND(AVG(value) * 100, 2) AS mortgage_rate_15yr_pct
  FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.FREDDIE_MAC_HOUSING_TIMESERIES
  WHERE variable_name = '15-Year Fixed Rate Mortgage Rate, National Average'
    AND geo_id = 'country/USA'
  GROUP BY 1
),
unemployment AS (
  SELECT DATE_TRUNC('month', date) AS month, geo_id, AVG(value) AS unemployment_rate_pct
  FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.BUREAU_OF_LABOR_STATISTICS_EMPLOYMENT_TIMESERIES
  WHERE variable_name = 'Local Area Unemployment: Unemployment Rate, Not seasonally adjusted, Monthly'
    AND LENGTH(geo_id) = 8
  GROUP BY 1, 2
),
national_unemployment AS (
  SELECT month, ROUND(AVG(unemployment_rate_pct), 2) AS unemployment_rate_pct
  FROM unemployment
  GROUP BY 1
),
income_raw AS (
  SELECT agi.geo_id, YEAR(agi.date) AS yr,
         ROUND(agi.value / NULLIF(ret.value, 0), 0) AS avg_income_per_return
  FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.IRS_INDIVIDUAL_INCOME_TIMESERIES agi
  JOIN SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.IRS_INDIVIDUAL_INCOME_TIMESERIES ret
    ON agi.geo_id = ret.geo_id AND agi.date = ret.date
  WHERE agi.variable_name = 'Adjusted gross income (AGI), AGI bin: Total'
    AND ret.variable_name = 'Number of returns, AGI bin: Total'
    AND LENGTH(agi.geo_id) = 8
),
income_indexed AS (
  SELECT geo_id, yr, avg_income_per_return,
         ROUND((avg_income_per_return /
                FIRST_VALUE(avg_income_per_return) OVER (PARTITION BY geo_id ORDER BY yr)) * 100, 1) AS income_index
  FROM income_raw
),
national_income AS (
  SELECT yr, ROUND(AVG(income_index), 1) AS income_index
  FROM income_indexed
  GROUP BY 1
),
geo AS (
  SELECT geo_id, geo_name
  FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.GEOGRAPHY_INDEX
  WHERE level = 'State'
),
national AS (
  SELECT c.month AS date, 'country/USA' AS geo_id, 'United States' AS geo_name,
         ROUND(c.cpi_index, 2) AS cpi_index,
         ROUND(((c.cpi_index - LAG(c.cpi_index, 12) OVER (ORDER BY c.month))
                / NULLIF(LAG(c.cpi_index, 12) OVER (ORDER BY c.month), 0)) * 100, 2) AS inflation_pct,
         m30.mortgage_rate_30yr_pct, m15.mortgage_rate_15yr_pct,
         nu.unemployment_rate_pct, ni.income_index
  FROM cpi c
  LEFT JOIN mortgage_30yr m30 ON c.month = m30.month
  LEFT JOIN mortgage_15yr m15 ON c.month = m15.month
  LEFT JOIN national_unemployment nu ON c.month = nu.month
  LEFT JOIN national_income ni ON YEAR(c.month) = ni.yr
),
states AS (
  SELECT u.month AS date, u.geo_id, g.geo_name,
         NULL::FLOAT AS cpi_index, NULL::FLOAT AS inflation_pct,
         NULL::FLOAT AS mortgage_rate_30yr_pct, NULL::FLOAT AS mortgage_rate_15yr_pct,
         u.unemployment_rate_pct, ii.income_index
  FROM unemployment u
  JOIN geo g ON u.geo_id = g.geo_id
  LEFT JOIN income_indexed ii ON u.geo_id = ii.geo_id AND YEAR(u.month) = ii.yr
)
SELECT * FROM national
UNION ALL
SELECT * FROM states
ORDER BY date, geo_id;

### Quick data profiling

Snowsight gives you profiling, charting, and pivot tables right in the query results. After running the query, try:
- **Chart** tab → line chart to visualize trends over time
- **Query Profile** tab → see the execution plan
- Column headers → quick profiling stats (min, max, distribution)

In [ ]:
-- National economic indicators since 2015
SELECT date, cpi_index, income_index, inflation_pct,
       mortgage_rate_30yr_pct, unemployment_rate_pct
FROM hol_db.public.economic_indicators
WHERE geo_id = 'country/USA'
  AND date >= '2015-01-01'
  AND inflation_pct IS NOT NULL
ORDER BY date;

## Step 4: Create the Semantic View

We have a clean table. Any analyst can query it with SQL, but that doesn't make it AI-ready. When an LLM sees column names like `CPI_INDEX` or `GEO_ID`, it guesses what they mean, and it can guess wrong.

A **Semantic View** tells the model which columns are dimensions, which are facts, how metrics are calculated, and what kinds of questions the table can answer. You define your business logic once, in the data layer, and every AI consumer inherits the same correct definitions.

**UI-Snowsight (optional, fastest way to get started):** AI & ML → Cortex Analyst → Create Semantic View.
- Select table: `HOL_DB.PUBLIC.ECONOMIC_INDICATORS`.
- Click **Autopilot** to auto-generate dimensions, facts, and metrics from the table schema.
- Review the generated YAML, add descriptions, and optionally add verified queries (known-good question → SQL pairs).
- Save as `economic_semantic_view`.

In this lab we create it via code (YAML below).

In [ ]:
CALL SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML(
  'hol_db.public',
  $$
name: economic_semantic_view
tables:
  - name: economic_indicators
    base_table:
      database: HOL_DB
      schema: PUBLIC
      table: ECONOMIC_INDICATORS
    dimensions:
      - name: DATE
        description: "Date of the observation"
        expr: economic_indicators.DATE
        data_type: DATE
      - name: GEO_ID
        description: "Geographic area identifier"
        expr: economic_indicators.GEO_ID
        data_type: TEXT
      - name: GEO_NAME
        description: "Geographic area — United States for national, or state name (e.g. California)"
        expr: economic_indicators.GEO_NAME
        data_type: TEXT
    facts:
      - name: CPI_INDEX
        description: "Consumer Price Index, base period 1982-84 = 100 (national only)"
        expr: economic_indicators.CPI_INDEX
        data_type: NUMBER
      - name: INFLATION_PCT
        description: "Year-over-year inflation rate as percent (national only)"
        expr: economic_indicators.INFLATION_PCT
        data_type: NUMBER
      - name: MORTGAGE_RATE_30YR_PCT
        description: "30-year fixed mortgage rate, national average, percent (national only)"
        expr: economic_indicators.MORTGAGE_RATE_30YR_PCT
        data_type: NUMBER
      - name: MORTGAGE_RATE_15YR_PCT
        description: "15-year fixed mortgage rate, national average, percent (national only)"
        expr: economic_indicators.MORTGAGE_RATE_15YR_PCT
        data_type: NUMBER
      - name: UNEMPLOYMENT_RATE_PCT
        description: "Unemployment rate as percent (available national and by state)"
        expr: economic_indicators.UNEMPLOYMENT_RATE_PCT
        data_type: NUMBER
      - name: INCOME_INDEX
        description: "Average income per tax return, indexed to earliest available year = 100. Compare to CPI_INDEX to assess purchasing power. (available national and by state, annual grain)"
        expr: economic_indicators.INCOME_INDEX
        data_type: NUMBER
    metrics:
      - name: AVG_CPI_INDEX
        description: "Average Consumer Price Index"
        expr: AVG(economic_indicators.CPI_INDEX)
      - name: AVG_INFLATION_PCT
        description: "Average year-over-year inflation rate"
        expr: AVG(economic_indicators.INFLATION_PCT)
      - name: AVG_MORTGAGE_RATE_30YR
        description: "Average 30-year fixed mortgage rate"
        expr: AVG(economic_indicators.MORTGAGE_RATE_30YR_PCT)
      - name: AVG_UNEMPLOYMENT_RATE
        description: "Average unemployment rate"
        expr: AVG(economic_indicators.UNEMPLOYMENT_RATE_PCT)
      - name: AVG_INCOME_INDEX
        description: "Average income index"
        expr: AVG(economic_indicators.INCOME_INDEX)
  $$
);

-- Verify
SHOW SEMANTIC VIEWS IN SCHEMA hol_db.public;

## Step 5: Create the Cortex Agent

Now we wrap the Semantic View in a conversational interface. A **Cortex Agent** takes a natural-language question and passes it to Cortex Analyst. Cortex Analyst uses the Semantic View to generate correct SQL, executes it, and returns a grounded answer with supporting data.

The whole thing is defined in a single SQL statement, so it's reproducible and version-controllable.

**UI-Snowsight (optional alternative):** AI & ML → Cortex Agents → **+ Create Agent**.
- Name: `hol_economic_agent`
- Tools: add a Cortex Analyst tool → select `economic_semantic_view`
- Instructions: "Answer questions about US economic indicators including inflation, mortgage rates, unemployment by state, and income trends."
- Warehouse: `hol_wh`
- Test the agent in the preview pane before saving.

In this lab we create it via SQL. The statement also grants the consumer role everything it needs to use the agent.

In [ ]:
-- Create a Cortex Agent backed by the economic indicators semantic view
CREATE OR REPLACE AGENT hol_db.public.hol_economic_agent
FROM SPECIFICATION
$$
tools:
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: economic_analyst
      description: "Answers questions about US economic indicators: CPI/inflation, mortgage interest rates (30-year, 15-year), unemployment rate (national and by state), and income index."
tool_resources:
  economic_analyst:
    semantic_view: HOL_DB.PUBLIC.ECONOMIC_SEMANTIC_VIEW
    execution_environment:
      type: warehouse
      warehouse: HOL_WH
$$;

-- Grant consumer role usage on the agent
GRANT USAGE  ON AGENT hol_db.public.hol_economic_agent TO ROLE end_user_role;
GRANT SELECT ON SEMANTIC VIEW hol_db.public.economic_semantic_view TO ROLE end_user_role;
GRANT SELECT ON TABLE hol_db.public.economic_indicators TO ROLE end_user_role;

-- Verify
SHOW AGENTS IN SCHEMA hol_db.public;

## Step 6: Chat with the Agent in Snowflake CoWork

Snowflake CoWork is the chat surface for business users: no SQL knowledge needed, no worksheet, just a conversation with the agent. Let's switch to `end_user_role` to see what it looks like for someone who can only consume, not build.

**UI-Snowsight:** AI & ML → **Open Snowflake CoWork**.
1. In CoWork, go to the bottom-left profile → **Settings** → switch role to `end_user_role` and warehouse to `hol_wh`.
2. Confirm `hol_economic_agent` appears in the agent list (control buttons of the CoWork chat).
3. Try these questions:
   - *How has the 30-year mortgage rate changed relative to inflation since 2020?*
   - *Tell the economic story of California vs Texas over the last 10 years using all available indicators.*
   - *What did the COVID shock look like in data — unemployment spike, rate crash, inflation surge — and how long did each phase last?*

The responses include the generated SQL, so you can see exactly which queries were executed. Same agent, same data, different role: a chat-based surface instead of SQL.

## Cleanup

Run only when you're done with the lab.

In [ ]:
USE ROLE ACCOUNTADMIN;

-- Drop every object the workshop roles own BEFORE dropping the roles themselves.
-- Drop database first (cascades all objects inside: tables, semantic views, agents)
DROP DATABASE IF EXISTS hol_db;
DROP WAREHOUSE IF EXISTS hol_wh;

-- Roles last, once nothing they own is left
DROP ROLE IF EXISTS hol_role;
DROP ROLE IF EXISTS end_user_role;

SHOW ROLES LIKE '%HOL%';